# Checking output files

In [2]:
from datetime import datetime, timedelta
import calendar
import collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import sys

import glob
import pandas as pd
import xarray as xr
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


my_dir = "/g/data/eg3/spr548/projects/"
sys.path.append(os.path.join(my_dir, "nesp_bff")+os.sep)
from nathers import location_details

# import utils
# from utils import locations, model_dict, vars_1hr, vars_day

In [34]:
step = "step3"
show_only_if_flagged = False

#==========================================
root_dir = "/g/data/eg3/nesp_bff/"
# location = "Sydney"
# scenario = "ssp370"
# model = "CESM2"
# time_period = "2041-2060"
vars_to_summarise_step2 = ["tas","huss","sfcWind","psl","rsds","rsdsdir","rsdsdif"]
vars_to_summarise_step3 = ["tas","twbt","huss","psl","wind_speed","wind_dir","clt","rsds","rsdsdir","rsdsdif"]
locations = ["Darwin","Cairns","Brisbane","Longreach","Mildura","Adelaide","Perth","Sydney","Melbourne","Canberra","Hobart"]
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]

input_dir = f"{root_dir}step3_calc_missing_vars/" if step == "step3" else f"{root_dir}step2_qdc_scaling/BARPA-R/"
vars_to_summarise = vars_to_summarise_step3 if step == "step3" else vars_to_summarise_step2

In [35]:
files = glob.glob(f"{input_dir}*.nc")
len(files)

42

In [38]:
# # Old checkds
# qc_rows = []
# errors = []

# for file in sorted(files):
#     base = file.split("/")[-1]
#     parts = base.split("_")

#     loc = parts[0] if len(parts) > 0 else None
#     model_ = parts[2] if len(parts) > 2 else None
#     ssp = parts[3] if len(parts) > 3 else None
#     time_period_ = parts[9] if len(parts) > 9 else None

#     header = f"{loc}: {model_}, {ssp}, {time_period_}"
#     print(f"==================== {header} ====================")

#     try:
#         with xr.open_dataset(file) as da:
#             df = (
#                 da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
#                   [vars_to_summarise]
#                   .to_dataframe()
#             )

#         desc = df.describe()

#         flagged = df.isna().any().any() or (df.nunique(dropna=True) <= 1).any()

#         if (not show_only_if_flagged) or flagged:
#             print("⚠️ Flagged" if flagged else "OK")
#             display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
#             display(desc)
#             display(HTML("</div>"))

#         qc_rows.append({
#             "file": base,
#             "loc": loc,
#             "model": model_,
#             "ssp": ssp,
#             "time_period": time_period_,
#             "n_min": int(desc.loc["count"].min()),
#             "any_nan": bool(df.isna().any().any()),
#             "any_const": bool((df.nunique(dropna=True) <= 1).any()),
#             "rsds_min": float(desc.loc["min", "rsds"]) if "rsds" in desc.columns else None,
#             "rsds_max": float(desc.loc["max", "rsds"]) if "rsds" in desc.columns else None,
#         })

#     except Exception as e:
#         errors.append({
#             "file": base,
#             "loc": loc,
#             "model": model_,
#             "ssp": ssp,
#             "time_period": time_period_,
#             "error": repr(e),
#         })

# qc_df = pd.DataFrame(qc_rows)
# err_df = pd.DataFrame(errors)

# print("\n=== QC summary (all files) ===")
# display(qc_df)

# if not err_df.empty:
#     print("\n=== Errors ===")
#     display(err_df)

In [40]:
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]
show_only_if_flagged = True
RANGES = {
   # temps: keep the K→C heuristic because files might still be in K
   "tas":  {"min": -60, "max":  60, "units": "C_or_K"},
   "twbt": {"min": -60, "max":  60, "units": "C_or_K"},
   # specific humidity (g/kg)
   "huss": {"min": 0.0, "max": 40., "units": "g/kg"},
   # sea level pressure in kPa (typical ~ 90–110 kPa; give generous bounds)
   "psl":  {"min": 80.0, "max": 110.0, "units": "kPa"},
   # wind speed (m/s)
   "wind_speed": {"min": 0.0, "max": 60.0, "units": "m/s"},
   # 16-point direction code (0..16). If you treat 0 as calm, keep it allowed.
   "wind_dir": {"min": 0.0, "max": 16.0, "units": "dir16"},
   # cloud cover in oktas (0..8)
   "clt": {"min": 0.0, "max": 8.0, "units": "oktas"},
   # radiation (W/m²)
   "rsds":    {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdir": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
   "rsdsdif": {"min": 0.0, "max": 1300.0, "units": "W/m2"},
}

def _convert_for_check(var, vmin, vmax):
    rule = RANGES.get(var)
    if rule is None:
        return vmin, vmax, None, None, None
    rmin, rmax = rule["min"], rule["max"]
    note = None
    # Temperature heuristic: if values look like Kelvin, convert to Celsius for checking
    if var in ("tas", "twbt"):
        if np.isfinite(vmax) and vmax > 150:  # likely Kelvin
            vmin = vmin - 273.15
            vmax = vmax - 273.15
            note = "converted K→C (heuristic)"
    return vmin, vmax, rmin, rmax, note

def flag_df(df, desc, vars_to_check):
   reasons = []
   # NaN + constant checks (overall)
   if df[vars_to_check].isna().any().any():
       reasons.append("has NaNs")
   if (df[vars_to_check].nunique(dropna=True) <= 1).any():
       const_vars = list(df[vars_to_check].columns[(df[vars_to_check].nunique(dropna=True) <= 1)])
       reasons.append(f"constant: {const_vars[:5]}" + ("..." if len(const_vars) > 5 else ""))
   # Range checks per variable
   for v in vars_to_check:
       if v not in desc.columns:
           reasons.append(f"missing column: {v}")
           continue
       vmin = desc.loc["min", v]
       vmax = desc.loc["max", v]
       vmin2, vmax2, rmin, rmax, note = _convert_for_check(v, vmin, vmax)
       if rmin is None:
           continue
       out_low = np.isfinite(vmin2) and (vmin2 < rmin)
       out_high = np.isfinite(vmax2) and (vmax2 > rmax)
       if out_low or out_high:
           msg = f"{v} out of range [{rmin}, {rmax}]"
           if out_low:
               msg += f" (min={vmin2:.3g})"
           if out_high:
               msg += f" (max={vmax2:.3g})"
           if note:
               msg += f" [{note}]"
           reasons.append(msg)
   return reasons

qc_rows, errors = [], []
for file in sorted(files):
   base = file.split("/")[-1]
   parts = base.split("_")
   loc = parts[0] if len(parts) > 0 else None
   model_ = parts[2] if len(parts) > 2 else None
   ssp = parts[3] if len(parts) > 3 else None
   time_period_ = parts[9] if len(parts) > 9 else None
   header = f"{loc}: {model_}, {ssp}, {time_period_}"
   print(f"==================== {header} ====================")
   try:
       with xr.open_dataset(file) as da:
           df = (
               da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
                 [vars_to_summarise]
                 .to_dataframe()
           )
       desc = df.describe()
       reasons = flag_df(df, desc, vars_to_summarise)
       flagged = len(reasons) > 0
       if (not show_only_if_flagged) or flagged:
           print("⚠️ Flagged" if flagged else "OK")
           if flagged:
               print("Reasons:", "; ".join(reasons[:6]) + (" ..." if len(reasons) > 6 else ""))
           display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
           display(desc)
           display(HTML("</div>"))
       qc_rows.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "flagged": flagged,
           "flag_reason": "; ".join(reasons[:8]) + (" ..." if len(reasons) > 8 else ""),
           "n_min": int(desc.loc["count"].min()),
           "any_nan": bool(df.isna().any().any()),
           "any_const": bool((df.nunique(dropna=True) <= 1).any()),
       })
   except Exception as e:
       errors.append({
           "file": base,
           "loc": loc,
           "model": model_,
           "ssp": ssp,
           "time_period": time_period_,
           "error": repr(e),
       })
qc_df = pd.DataFrame(qc_rows)
err_df = pd.DataFrame(errors)
display(qc_df.sort_values(["flagged", "loc", "model"], ascending=[False, True, True]))
if not err_df.empty:
   display(err_df)

==================== Melbourne: ACCESS-CM2, ssp126, 2021-2040 ====================
==================== Melbourne: ACCESS-CM2, ssp126, 2041-2060 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=60.7)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,15.531325,11.811526,7.376869,100.381027,5.033895,11.070873,5.530114,178.273987,197.229126,60.457741
std,6.198773,4.013904,2.264378,0.749708,2.951959,4.549204,2.366922,267.716064,260.492004,83.693253
min,-0.103054,-0.651840,1.010355,96.607674,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,11.161758,8.897761,5.739625,99.915327,2.932193,8.000000,3.000000,0.000000,0.000000,0.000000
50%,14.571415,11.420112,6.927511,100.386803,4.557750,12.000000,6.000000,5.000000,0.000000,3.533095
75%,18.865087,14.537907,8.583199,100.869034,6.734211,16.000000,8.000000,292.145226,386.395157,108.606462
max,46.054981,31.133121,22.816242,102.893982,60.722473,16.000000,8.000000,1160.785400,1199.771606,384.553467


==================== Melbourne: ACCESS-CM2, ssp126, 2061-2080 ====================
==================== Melbourne: ACCESS-CM2, ssp370, 2021-2040 ====================
==================== Melbourne: ACCESS-CM2, ssp370, 2041-2060 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=62.4)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,16.102825,12.261433,7.584716,100.425217,5.033574,11.070873,5.530114,179.911774,201.527191,59.421043
std,6.195697,3.982809,2.295161,0.750465,2.917476,4.549204,2.366922,269.445343,265.448059,81.616379
min,-0.330783,-0.876673,1.253946,96.768875,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,11.741392,9.379382,5.917593,99.958336,2.937562,8.000000,3.000000,0.000000,0.000000,0.000000
50%,15.206291,11.938084,7.141523,100.411938,4.572999,12.000000,6.000000,5.000000,0.000000,3.575004
75%,19.463557,14.981495,8.826676,100.919550,6.763319,16.000000,8.000000,295.530464,396.907028,107.881886
max,46.024250,30.911898,22.242794,102.889420,62.408310,16.000000,8.000000,1156.264282,1197.048340,386.100403


==================== Melbourne: ACCESS-CM2, ssp370, 2061-2080 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=67.2)


...

==================== Melbourne: NorESM2-MM, ssp370, 2041-2060 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=66.1)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,15.041800,11.347757,7.101709,100.347649,5.161282,11.070873,5.530114,176.746445,189.716003,63.148029
std,6.099096,3.860930,2.077054,0.763508,2.940399,4.549204,2.366922,266.916473,252.595154,88.273338
min,-0.059903,-0.608210,1.177436,96.837265,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.759281,8.572299,5.628664,99.851866,3.078647,8.000000,3.000000,0.000000,0.000000,0.000000
50%,14.091337,11.005273,6.708224,100.372940,4.748676,12.000000,6.000000,5.000000,0.000000,3.712749
75%,18.238458,13.955603,8.165133,100.886909,6.880485,16.000000,8.000000,287.023048,366.506088,112.324398
max,47.338150,29.951212,21.770842,102.732330,66.084068,16.000000,8.000000,1159.131348,1158.988525,396.887817


==================== Melbourne: NorESM2-MM, ssp370, 2061-2080 ====================
⚠️ Flagged
Reasons: wind_speed out of range [0.0, 60.0] (max=69.6)


,tas,twbt,huss,psl,wind_speed,wind_dir,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000
mean,15.647951,11.916795,7.420563,100.396996,5.150136,11.070873,5.530114,178.249481,193.739380,62.320957
std,6.250585,3.931626,2.201454,0.766026,2.931838,4.549204,2.366922,268.421844,256.832977,86.473854
min,0.349574,-0.158930,1.150941,96.785912,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,11.286812,9.076270,5.837569,99.897160,3.045976,8.000000,3.000000,0.000000,0.000000,0.000000
50%,14.642450,11.563823,6.997269,100.420929,4.751136,12.000000,6.000000,5.000000,0.000000,3.703393
75%,18.821293,14.570474,8.567862,100.928757,6.880580,16.000000,8.000000,291.056564,377.208351,111.947289
max,47.247547,29.888203,21.429182,102.843071,69.648834,16.000000,8.000000,1154.170898,1132.278076,395.328644


,file,loc,model,ssp,time_period,flagged,flag_reason,n_min,any_nan,any_const
1,Melbourne_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BO...,Melbourne,ACCESS-CM2,ssp126,2041-2060,True,"wind_speed out of range [0.0, 60.0] (max=60.7)",175200,False,False
4,Melbourne_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BO...,Melbourne,ACCESS-CM2,ssp370,2041-2060,True,"wind_speed out of range [0.0, 60.0] (max=62.4)",175200,False,False
5,Melbourne_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BO...,Melbourne,ACCESS-CM2,ssp370,2061-2080,True,"wind_speed out of range [0.0, 60.0] (max=67.2)",175200,False,False
6,Melbourne_AUS-15_ACCESS-ESM1-5_ssp126_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp126,2021-2040,True,"wind_speed out of range [0.0, 60.0] (max=63.1)",175200,False,False
7,Melbourne_AUS-15_ACCESS-ESM1-5_ssp126_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp126,2041-2060,True,"wind_speed out of range [0.0, 60.0] (max=66)",175200,False,False
8,Melbourne_AUS-15_ACCESS-ESM1-5_ssp126_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp126,2061-2080,True,"wind_speed out of range [0.0, 60.0] (max=69.2)",175200,False,False
9,Melbourne_AUS-15_ACCESS-ESM1-5_ssp370_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp370,2021-2040,True,"wind_speed out of range [0.0, 60.0] (max=65.3)",175200,False,False
10,Melbourne_AUS-15_ACCESS-ESM1-5_ssp370_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp370,2041-2060,True,"wind_speed out of range [0.0, 60.0] (max=63.7)",175200,False,False
11,Melbourne_AUS-15_ACCESS-ESM1-5_ssp370_r6i1p1f1...,Melbourne,ACCESS-ESM1-5,ssp370,2061-2080,True,"wind_speed out of range [0.0, 60.0] (max=66.4)",175200,False,False
12,Melbourne_AUS-15_CESM2_ssp126_r11i1p1f1_BOM_BA...,Melbourne,CESM2,ssp126,2021-2040,True,"wind_speed out of range [0.0, 60.0] (max=63.3)",175200,False,False


In [39]:
qc_rows

[]